# Solar Filament V11 — CPU Learned Fragment-Graph Repair

**A different hypothesis from V4–V10.** Learn whether two separately predicted masks describe *the same physical filament*, using annotator-labeled TRAIN images only. Merge fragments when a calibrated pairwise graph supports the join. Unlike pixel-level boosting, this directly targets the **one-to-many fragmentation** error of Panoptic Quality.

**No GPU; no opaque 0.55 CSV; no use of test labels. No score guaranteed.** Existing V4/V7 weights are mandatory. The best known public score remains **0.34**; V10 scored 0.33 because its final export was a *reconstructed V7 baseline*, not its learned quality model. If graph repair fails a frozen validation gate, the notebook writes diagnostics **without a new submission** by default.

Modes: `FIT` (default; train graph / select / diagnose) → save output dataset → `SUBMIT` only if selected gain is substantial. `ALL` performs both when promoted. Exact original 0.34 pipeline cannot be guaranteed without its original V7 `selection.json` or winning CSV; this notebook reports that distinction explicitly.

Inspiration: filament skeleton connectivity, instance graph grouping, and topology-aware segmentation. The method here is an original constrained pairwise classifier, not a reproduction of any named paper.

References: https://www.kaggle.com/competitions/filament-segmentation-2026 and https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook .

In [ ]:
import os,sys,json,hashlib,random,subprocess,importlib.util,gc,math,shutil
from pathlib import Path
from collections import defaultdict
from itertools import product
import numpy as np,pandas as pd,cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from IPython.display import display

# CHANGE ONLY THIS CELL
MODE='FIT'                      # FIT | SUBMIT | ALL. Prefer FIT first.
DATA_ROOT=Path('/kaggle/input/datasets/lighittha/solar3k/MAGFiLO_1.0_Kaggle_2026')
V4_ROOT=Path('/kaggle/input/datasets/lighittha/solaroutsp3/solar_cpu_v4')
V7_ROOT=None                    # If known, folder with v7_config.json, selection.json, tiles/
PREVIOUS_V11_ROOT=None          # Attached earlier /solar_graph_v11 output
OUT=Path('/kaggle/working/solar_graph_v11' if Path('/kaggle/working').exists() else './solar_graph_v11')
SEED=2026; H=W=2048
GRAPH_TRAIN_IMAGES=160          # V4 original TRAIN images only; raise to 350 if results are encouraging
MAX_PAIR_GAP=85                 # physical pixels: consider pairs with bbox gap <=85
MIN_AREA=350
MIN_ABSOLUTE_GAIN=.015         # PQ gain on reserved 47-image tune selection before export
ALLOW_BASELINE_EXPORT=False    # False avoids another pointless 0.33-ish submission
TRAIN_FRACTION_FOR_BASELINE=.52 # original tune split: ~50 fit recipe / 47 select graph
RAW_CONF=.10; RAW_IOU=.82; GLOBAL_IMGSZ=1024; TILE_SIDE=1280; TILE_IMGSZ=1024
MAX_DET=65; AUTO_INSTALL=True

OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);cv2.setNumThreads(0)
for m,p in [('pycocotools','pycocotools'),('ultralytics','ultralytics>=8.3,<9'),('joblib','joblib')]:
    if importlib.util.find_spec(m) is None:
        if not AUTO_INSTALL:raise ImportError(p)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
import torch,joblib
from pycocotools import mask as coco
from ultralytics import YOLO
if hasattr(torch,'set_num_threads'):torch.set_num_threads(min(4,os.cpu_count() or 2))
print('V11 graph repair, CPU, mode',MODE,'out',OUT)

## 1 · Data integrity, fixed grouped split, and checkpoint provenance

In [ ]:
def locate_root(root,marker,extra=None):
    if root is not None:
        p=Path(root)
        if (p/marker).is_file() and (extra is None or (p/extra).exists()):return p
    bases=[Path('/kaggle/input'),Path('/mnt/data')]
    roots=[]
    for base in bases:
        if base.exists():
            for p in base.rglob(Path(marker).name):
                if len(Path(marker).parts)==1 and p.name==marker: parent=p.parent
                elif str(p).endswith(marker):parent=p.parents[len(Path(marker).parts)-1]
                else:continue
                if extra is None or (parent/extra).exists():roots.append(parent)
    if not roots:raise FileNotFoundError(f'Cannot locate {marker} and {extra}. Attach original dataset.')
    roots=sorted(set(roots),key=str)
    print('Auto-found asset',marker,roots[0]);return roots[0]

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for part in iter(lambda:f.read(1<<20),b''):h.update(part)
    return h.hexdigest()

DATA=locate_root(DATA_ROOT,'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json','test/test_images')
V4=locate_root(V4_ROOT,'split_manifest.json','runs/yolo11n-seg/weights/best.pt')
CKPT=V4/'runs/yolo11n-seg/weights/best.pt'; CKPT_SHA=sha(CKPT)
SPLIT=json.loads((V4/'split_manifest.json').read_text()); SPLIT_SHA=sha(V4/'split_manifest.json')
TRAIN={p.stem:p for p in (DATA/'train/train_images').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')}
TEST={p.stem:p for p in (DATA/'test/test_images').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')}
assert len(TRAIN)==707 and len(TEST)==180
assert set().union(*(set(SPLIT[k]) for k in ('train','tune','audit')))==set(TRAIN)
for a in SPLIT:
    for b in SPLIT:
        if a!=b:assert not ({s[:8] for s in SPLIT[a]}&{s[:8] for s in SPLIT[b]})
raw=json.loads((DATA/'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
by_id=defaultdict(list)
for a in raw['annotations']:by_id[str(a['image_id'])].append(a)
OBS=defaultdict(list)
for meta in raw['images']:
    s=Path(meta['file_name']).stem
    if s in TRAIN:OBS[s].append((str(meta['id']),by_id[str(meta['id'])]))
assert set(OBS)==set(TRAIN)

v7_candidates=[]
for base in (Path('/kaggle/input'),Path('/mnt/data')):
    if base.exists():v7_candidates += [x.parent for x in base.rglob('v7_config.json')]
if V7_ROOT and (Path(V7_ROOT)/'v7_config.json').is_file():v7_candidates.insert(0,Path(V7_ROOT))
V7=None; V7CFG={}; SAVED_STRATEGY=None
for candidate in v7_candidates:
    try:
        obj=json.loads((candidate/'v7_config.json').read_text())
        if obj.get('checkpoint_sha256')!=CKPT_SHA or obj.get('split_sha256')!=SPLIT_SHA:continue
        V7=candidate;V7CFG=obj
        if (candidate/'selection.json').is_file():
            selection=json.loads((candidate/'selection.json').read_text())
            if selection.get('source_model_sha256')==CKPT_SHA and selection.get('split_sha256')==SPLIT_SHA:
                SAVED_STRATEGY=selection.get('strategy')
                break
    except Exception:continue
if V7CFG:
    TILE_SIDE=int(V7CFG.get('tile_side',TILE_SIDE));TILE_IMGSZ=int(V7CFG.get('tile_imgsz',TILE_IMGSZ))
TILE_SCHEME=V7CFG.get('tile_scheme','FOUR')
assert TILE_SCHEME in ('FOUR','FIVE')
TILE_WINDOWS=sorted(set([(0,0),(W-TILE_SIDE,0),(0,H-TILE_SIDE),(W-TILE_SIDE,H-TILE_SIDE)]+([((W-TILE_SIDE)//2,(H-TILE_SIDE)//2)] if TILE_SCHEME=='FIVE' else [])))
print('TRAIN/TUNE/AUDIT',*[len(SPLIT[k]) for k in ('train','tune','audit')],'V7:',V7,'exact saved recipe:',SAVED_STRATEGY is not None)
print('Model SHA',CKPT_SHA,'V7 windows:',TILE_WINDOWS)

## 2 · COCO mask utilities and one-to-one, per-annotator PQ

In [ ]:
def rle(s):
    if isinstance(s,dict):s=s['counts']
    return {'size':[H,W],'counts':s.encode('ascii') if isinstance(s,str) else s}
def asc(m):return m['counts'].decode('ascii') if isinstance(m['counts'],bytes) else str(m['counts'])
def enc(m):return coco.encode(np.asfortranarray(np.asarray(m,np.uint8)))
def gt_one(a):
    seg=a.get('segmentation',[])
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):return coco.frPyObjects(seg,H,W)
        return rle(seg)
    if not seg:return None
    if isinstance(seg[0],(int,float)):seg=[seg]
    pp=[]
    for vertices in seg:
        try:
            xyz=np.asarray(vertices,float).reshape(-1,2)
            if len(xyz)>=3 and np.isfinite(xyz).all():pp.append(np.clip(xyz,[0,0],[W-1,H-1]).ravel().tolist())
        except (TypeError,ValueError):continue
    if not pp:return None
    rr=coco.frPyObjects(pp,H,W)
    return coco.merge(rr) if isinstance(rr,list) else rr
GT={}
def gt(stem):
    if stem not in GT:GT[stem]=[(uid,[r for a in annotations if (r:=gt_one(a)) is not None]) for uid,annotations in OBS[stem]]
    return GT[stem]
def match(truth,preds):
    if not truth and not preds:return np.zeros(4,float)
    if not truth:return np.array([0,0,len(preds),0],float)
    if not preds:return np.array([0,0,0,len(truth)],float)
    iou=np.asarray(coco.iou(truth,preds,[0]*len(preds)),float)
    rows,cols=linear_sum_assignment(-iou)
    keep=iou[rows,cols]>.5
    tp=int(keep.sum())
    return np.array([float(iou[rows[keep],cols[keep]].sum()),tp,len(preds)-tp,len(truth)-tp],float)
def stats(v):
    si,tp,fp,fn=map(float,v);den=tp+.5*(fp+fn)
    return {'pq':float(si/den) if den else 0.,'sq':float(si/tp) if tp else 0.,
            'rq':float(tp/den) if den else 0.,'tp':int(tp),'fp':int(fp),'fn':int(fn)}
def score(stems,fn):
    counts=np.zeros(4,float)
    for stem in stems:
        preds=fn(stem)
        for _,truth in gt(stem):counts+=match(truth,preds)
    return stats(counts)
def disjoint(rs):
    if len(rs)<2:return True
    pair=np.asarray(coco.iou(rs,rs,[0]*len(rs)))
    np.fill_diagonal(pair,0)
    return bool(np.all(pair==0))
unit=np.zeros((H,W),np.uint8);unit[20:100,15:95]=1
assert abs(stats(match([enc(unit)],[enc(unit)]))['pq']-1)<1e-8
print('PQ/RLE tests passed')

## 3 · Live image proposals with cache recovery
No precompiled test results. Cached proposals must match the exact source checkpoint and geometry; otherwise generate them from official JPEG pixels on CPU.

In [ ]:
P=OUT/'proposals';P.mkdir(exist_ok=True)
PREV=Path(PREVIOUS_V11_ROOT) if PREVIOUS_V11_ROOT else None
if PREV is None:
    possibles=[]
    for base in (Path('/kaggle/input'),):
        if base.exists():possibles.extend([p.parent for p in base.rglob('v11_contract.json')])
    for p in possibles:
        try:
            obj=json.loads((p/'v11_contract.json').read_text())
            if obj['checkpoint_sha256']==CKPT_SHA and obj['split_sha256']==SPLIT_SHA and obj['tile_scheme']==TILE_SCHEME:
                PREV=p;break
        except Exception:pass
CONTRACT={'checkpoint_sha256':CKPT_SHA,'split_sha256':SPLIT_SHA,'tile_side':TILE_SIDE,
          'tile_imgsz':TILE_IMGSZ,'tile_scheme':TILE_SCHEME,'raw_conf':RAW_CONF,'raw_iou':RAW_IOU,'v11':'graph-1'}
(OUT/'v11_contract.json').write_text(json.dumps(CONTRACT,indent=2))
NET=None
def getnet():
    global NET
    if NET is None:NET=YOLO(str(CKPT))
    return NET

def cache_path(kind,stem):return P/f'{kind}__{stem}.json'
def decode_json_meta(kind,stem):
    return {'stem':stem,'kind':kind,'checkpoint_sha256':CKPT_SHA,'tile_side':TILE_SIDE,
            'tile_imgsz':TILE_IMGSZ,'tile_scheme':TILE_SCHEME,'global_imgsz':GLOBAL_IMGSZ,
            'raw_conf':RAW_CONF,'raw_iou':RAW_IOU,'source':'LIVE_H_ALPHA_INFERENCE'}
def read_props(kind,stem):
    files=[cache_path(kind,stem)]
    if PREV:files.append(PREV/'proposals'/f'{kind}__{stem}.json')
    for p in files:
        if not p.is_file():continue
        try:
            obj=json.loads(p.read_text())
            if obj.get('meta')==decode_json_meta(kind,stem):return obj['predictions']
        except Exception:pass
    if kind=='global':
        f=V4/'cache/yolo11n-seg'/(stem+'.json')
        if f.is_file():
            try:
                d=json.loads(f.read_text())
                if d.get('source')=='LIVE_IMAGE_INFERENCE' and d.get('stem')==stem and d.get('checkpoint_sha256')==CKPT_SHA and d.get('predict_imgsz')==GLOBAL_IMGSZ and d.get('tta') is False:return d['proposals']
            except Exception:pass
    if kind=='tiles' and V7:
        f=V7/'tiles'/(stem+'.json')
        if f.is_file():
            try:
                d=json.loads(f.read_text())
                if d.get('stem')==stem and d.get('checkpoint_sha256')==CKPT_SHA and d.get('kind')=='tile' and d.get('source')=='LIVE_TEST_OR_TRAIN_JPEG_INFERENCE' and d.get('tile_side')==TILE_SIDE and d.get('tile_imgsz')==TILE_IMGSZ and d.get('tile_scheme')==TILE_SCHEME:
                    return d['proposals']
            except Exception:pass
    return None

@torch.inference_mode()
def infer_view(gray,kind,x=0,y=0):
    im=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR)
    pred=getnet().predict(source=im,imgsz=GLOBAL_IMGSZ if kind=='global' else TILE_IMGSZ,
            conf=RAW_CONF,iou=RAW_IOU,retina_masks=True,agnostic_nms=True,
            max_det=MAX_DET,device='cpu',verbose=False)[0]
    if pred.masks is None:return []
    ms=pred.masks.data.detach().cpu().numpy()
    ps=pred.boxes.conf.detach().cpu().numpy()
    out=[]
    for mask,confidence in zip(ms,ps):
        local=np.uint8(mask>.5)
        if local.shape!=gray.shape:local=cv2.resize(local,(gray.shape[1],gray.shape[0]),interpolation=cv2.INTER_NEAREST)
        if not local.any():continue
        if kind=='tiles':
            edge=bool((x>0 and local[:,:12].any()) or (y>0 and local[:12,:].any()) or
                  (x+TILE_SIDE<W and local[:,-12:].any()) or (y+TILE_SIDE<H and local[-12:,:].any()))
            full=np.zeros((H,W),np.uint8);full[y:y+TILE_SIDE,x:x+TILE_SIDE]=local
        else:full=local;edge=False
        rr=enc(full)
        out.append({'score':float(confidence),'counts':asc(rr),'area':int(coco.area(rr)),
                    'edge':edge,'origin':'tile' if kind=='tiles' else 'global'})
    return out

def proposals(kind,stem,lookup):
    got=read_props(kind,stem)
    if got is not None:return got
    image=cv2.imread(str(lookup[stem]),cv2.IMREAD_GRAYSCALE)
    if image is None or image.shape!=(H,W):raise RuntimeError('Missing image '+stem)
    if kind=='global':got=infer_view(image,'global')
    else:
        got=[]
        for x,y in TILE_WINDOWS:got+=infer_view(image[y:y+TILE_SIDE,x:x+TILE_SIDE],'tiles',x,y)
    path=cache_path(kind,stem)
    path.write_text(json.dumps({'meta':decode_json_meta(kind,stem),'predictions':got},separators=(',',':')))
    assert read_props(kind,stem) is not None, ('Cache roundtrip failed:',stem,kind)
    return got

## 4 · Restore V7, never silently substitute a different baseline

If the exact V7 recipe isn't attached, reconstruct on **only the first 50 tune observations**; reserve the other 47 for graph selection. This is not automatically the original 0.34 strategy.

In [ ]:
TUNE=sorted(SPLIT['tune']);random.Random(SEED+392).shuffle(TUNE)
N_BASE=int(round(len(TUNE)*TRAIN_FRACTION_FOR_BASELINE))
BASE_FIT=sorted(TUNE[:N_BASE]);SELECT=sorted(TUNE[N_BASE:])
assert not set(BASE_FIT)&set(SELECT)

# Match earlier V7 disjoint selection policy.
def filter_props(items,conf_global=.32,conf_tile=.48,bonus=.0,area=350,guard=True,nms=.5,containment=.88,min_free=.50):
    pool=[]
    for obj in items:
        tile=obj.get('origin','global')=='tile'
        if float(obj['score'])<(conf_tile if tile else conf_global) or int(obj['area'])<area:continue
        if guard and tile and obj.get('edge',obj.get('truncated',False)):continue
        pool.append((float(obj['score'])+(bonus if tile else 0),obj))
    pool.sort(key=lambda x:(-x[0],-float(x[1]['area'])))
    occ=np.zeros((H,W),np.uint8);original=[];result=[]
    for _,obj in pool:
        rr=rle(obj['counts'])
        if original:
            s=np.asarray(coco.iou([rr],original,[0]*len(original)))[0]
            sizes=np.asarray(coco.area(original),float)
            intersection=s*(sizes+float(obj['area']))/(1+s+1e-10)
            if np.any(s>=nms) or np.any(intersection/max(1,obj['area'])>=containment):continue
        mask=coco.decode(rr).astype(bool)
        free=mask&(occ==0)
        if free.sum()<area or free.sum()/max(1,obj['area'])<min_free:continue
        occ[free]=1;original.append(rr);result.append(enc(free))
    assert disjoint(result)
    return result

def v4_control(stem,lookup):
    ps=proposals('global',stem,lookup)
    entries=sorted([p for p in ps if p['score']>=.32 and p['area']>=350],key=lambda x:(-x['score'],-x['area']))
    occ=np.zeros((H,W),np.uint8);prior=[];out=[]
    for obj in entries:
        rr=rle(obj['counts'])
        if prior and np.asarray(coco.iou([rr],prior,[0]*len(prior)))[0].max()>=.5:continue
        prior.append(rr);mask=coco.decode(rr).astype(bool)&(occ==0)
        if mask.sum()<350:continue
        occ[mask]=1;out.append(enc(mask))
    return out

def v7_control(stem,lookup,p):
    if p=='v4':return v4_control(stem,lookup)
    items=[dict(q,origin='global',edge=False) for q in proposals('global',stem,lookup)]
    if p['mode']!='global':items += [dict(q,origin='tile',edge=bool(q.get('edge',q.get('truncated',False)))) for q in proposals('tiles',stem,lookup)]
    return filter_props(items,conf_global=p['global_conf'],conf_tile=p['tile_conf'],bonus=p['tile_bonus'],
                        area=p['min_area'],guard=p['seam_guard'],nms=p['nms_iou'],containment=p['containment'],min_free=p['min_free_fraction'])

V7_GRID=[dict(mode='tiles',global_conf=gc,tile_conf=tc,tile_bonus=bo,min_area=350,
              seam_guard=guard,nms_iou=.5,containment=.88,min_free_fraction=.5)
         for gc,tc,bo,guard in product((.28,.36),(.35,.48,.62),(0.,.10),(False,True))]
V7_STRATEGY_FILE=OUT/'v7_baseline.json'
if MODE=='SUBMIT':
    src=V7_STRATEGY_FILE if V7_STRATEGY_FILE.is_file() else PREV/'v7_baseline.json' if PREV else None
    if src is None or not src.is_file():raise FileNotFoundError('Missing frozen V7 baseline, run FIT first')
    V7_PICK=json.loads(src.read_text())
else:
    if SAVED_STRATEGY is not None:
        base_cfg=SAVED_STRATEGY; identity='restored_exact_V7_recipe'
    else:
        print('Reconstruct V7 on',len(BASE_FIT),'recipe-fit observations; NOT exact 0.34 CSV')
        for s in tqdm(BASE_FIT,desc='Load CPU V7 proposals'):
            proposals('global',s,TRAIN);proposals('tiles',s,TRAIN)
        best=score(BASE_FIT,lambda s:v4_control(s,TRAIN))['pq'];base_cfg='v4'
        for cfg in tqdm(V7_GRID,desc='Reconstruct V7 recipe'):
            pq=score(BASE_FIT,lambda s:v7_control(s,TRAIN,cfg))['pq']
            if pq>best:best=pq;base_cfg=cfg
        identity='reconstructed_V7_not_original_0.34' if base_cfg!='v4' else 'V4_only'
        print('Recipe fit PQ',best)
    V7_PICK={'strategy':base_cfg,'identity':identity,'v4sha':CKPT_SHA,'splitsha':SPLIT_SHA}
    V7_STRATEGY_FILE.write_text(json.dumps(V7_PICK,indent=2))
assert V7_PICK['v4sha']==CKPT_SHA and V7_PICK['splitsha']==SPLIT_SHA
BASE_MEM={}
def baseline(stem,lookup):
    key=(stem,'train' if lookup is TRAIN else 'test')
    if key not in BASE_MEM:BASE_MEM[key]=v7_control(stem,lookup,V7_PICK['strategy'])
    return BASE_MEM[key]
print('Baseline identity:',V7_PICK['identity'],'graph-selection images:',len(SELECT))

## 5 · Pairwise fragment features: geometry, solar contrast, and gap morphology

Compute only nearby pairs (up to 85 native pixels between bounding boxes). Features assess shape, orientation, dark solar contrast along the proposed link, and distance between contour endpoints. This is **instance graph learning**, not pixel thresholding.

In [ ]:
FEAT_NAMES=['gap_px','gap_norm_size','log_small_area','log_large_area','area_ratio',
 'axis_parallel','join_alignment_a','join_alignment_b','dark_mean','dark_min','dark_against_region',
 'bbox_fill_a','bbox_fill_b','edge_distance','size_sum','bbox_center_distance']
PAIR_MEM={}
def sample_geometry(rr):
    box=np.asarray(coco.toBbox([rr]),float)[0]
    x,y,w,h=box
    local=coco.decode(rr).astype(np.uint8)
    rect=local[max(0,int(y)):min(H,int(y+h+1)),max(0,int(x)):min(W,int(x+w+1))]
    contours,_=cv2.findContours(rect,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not contours:return None
    pts=np.concatenate([c.reshape(-1,2) for c in contours if len(c)>0],axis=0).astype(np.float64)
    pts+=np.array([max(0,int(x)),max(0,int(y))])
    if len(pts)>350:pts=pts[np.linspace(0,len(pts)-1,350).astype(int)]
    center=pts.mean(0)
    v=pts-center
    if len(pts)>2:
        eigval,eigvec=np.linalg.eigh(v.T@v/max(1,len(v)))
        axis=eigvec[:,-1]
    else:axis=np.array([1.,0.])
    return {'pts':pts,'box':box,'axis':axis,'center':center,'area':float(coco.area(rr))}

def bbox_gap(a,b):
    ax,ay,aw,ah=a['box'];bx,by,bw,bh=b['box']
    dx=max(0.,bx-(ax+aw),ax-(bx+bw))
    dy=max(0.,by-(ay+ah),ay-(by+bh))
    return float(np.hypot(dx,dy))
def pair_features(A,B,gray):
    if bbox_gap(A,B)>MAX_PAIR_GAP:return None
    dist,index=cKDTree(B['pts']).query(A['pts'],k=1)
    i=int(np.argmin(dist));gap=float(dist[i]);j=int(index[i])
    if gap>MAX_PAIR_GAP:return None
    aa=A['pts'][i];bb=B['pts'][j]
    displacement=bb-aa
    norm=np.linalg.norm(displacement)
    direction=displacement/max(1.,norm)
    width=max(1.,np.sqrt(A['area'])+np.sqrt(B['area']))
    coords=np.linspace(0.,1.,23)[:,None]*displacement[None,:]+aa[None,:]
    xx=np.clip(np.rint(coords[:,0]).astype(int),0,W-1)
    yy=np.clip(np.rint(coords[:,1]).astype(int),0,H-1)
    values=gray[yy,xx].astype(float)
    mx,my=(aa+bb)/2
    y0=max(0,int(my)-14);y1=min(H,int(my)+15)
    x0=max(0,int(mx)-14);x1=min(W,int(mx)+15)
    bg=float(np.median(gray[y0:y1,x0:x1])) if y1>y0 and x1>x0 else float(values.mean())
    r1=np.linalg.norm(A['center']-[W/2,H/2]);r2=np.linalg.norm(B['center']-[W/2,H/2])
    ar=min(A['area'],B['area']); br=max(A['area'],B['area'])
    ax,ay,aw,ah=A['box']; bx,by,bw,bh=B['box']
    feats=[gap,gap/width,np.log1p(ar),np.log1p(br),ar/max(1,br),
           abs(float(np.dot(A['axis'],B['axis']))),abs(float(np.dot(A['axis'],direction))),
           abs(float(np.dot(B['axis'],direction))),float(values.mean()/255),float(values.min()/255),
           (bg-float(values.mean()))/64,float(A['area']/max(1,aw*ah)),float(B['area']/max(1,bw*bh)),
           min(r1,r2)/(H*.7071),np.log1p(A['area']+B['area']),
           np.linalg.norm(A['center']-B['center'])/(np.sqrt(A['area'])+np.sqrt(B['area'])+1.)]
    assert len(feats)==len(FEAT_NAMES) and np.isfinite(feats).all()
    return feats,(aa,bb)

def enumerate_pairs(stem,lookup):
    rs=baseline(stem,lookup)
    if len(rs)<2:return rs,[],[]
    geometries=[sample_geometry(v) for v in rs]
    gray=cv2.imread(str(lookup[stem]),cv2.IMREAD_GRAYSCALE)
    if gray is None:raise RuntimeError(stem)
    edges=[];features=[]
    for i in range(len(rs)):
        if geometries[i] is None:continue
        for j in range(i+1,len(rs)):
            if geometries[j] is None:continue
            found=pair_features(geometries[i],geometries[j],gray)
            if found is None:continue
            feats,anchors=found
            edges.append((i,j,anchors));features.append(feats)
    return rs,edges,features

# Synthetic geometry tests
u=np.zeros((H,W),np.uint8);u[120:135,100:195]=1
v=np.zeros((H,W),np.uint8);v[120:135,220:310]=1
A,B=sample_geometry(enc(u)),sample_geometry(enc(v))
assert A is not None and B is not None
assert len(pair_features(A,B,np.full((H,W),120,np.uint8))[0])==len(FEAT_NAMES)
print('Pair geometry/contrast tests passed')
del u,v

## 6 · Train graph classifier from original TRAIN images only

Pair labels are positive when both fragments belong to the same annotated GT filament and their union improves IoU by a meaningful amount. Multi-annotator votes reduce label noise. If too few positives exist, do not promote a graph model.

In [ ]:
def pair_truth(stem,rs,edges):
    if not edges:return np.empty(0,np.uint8)
    versions=gt(stem);positive=np.zeros(len(edges),float)
    for _,truth in versions:
        if not truth:continue
        ov=np.asarray(coco.iou(truth,rs,[0]*len(rs)),float)
        areas=np.asarray(coco.area(rs),float)
        garea=np.asarray(coco.area(truth),float)
        # Intersection from IoU for non-crowd GT
        ix=ov*(garea[:,None]+areas[None,:])/(1.+ov+1e-10)
        for k,(i,j,_) in enumerate(edges):
            frac_i=ix[:,i]/max(1,areas[i]);frac_j=ix[:,j]/max(1,areas[j])
            good=(frac_i>.46)&(frac_j>.46)
            if not good.any():continue
            join_inter=ix[:,i]+ix[:,j]
            join_iou=join_inter/np.maximum(1,garea+areas[i]+areas[j]-join_inter)
            if np.any(good&(join_iou>np.maximum(ov[:,i],ov[:,j])+.065)):
                positive[k]+=1
    return np.uint8(positive/max(1,len(versions))>=.5)

QFILE=OUT/'pair_graph_model.pkl';QMETA=OUT/'pair_graph_metadata.json';PAIRS_CSV=OUT/'train_pair_diagnostics.csv'
if MODE in ('FIT','ALL'):
    stems=sorted(SPLIT['train']); random.Random(SEED+72).shuffle(stems)
    stems=stems[:min(len(stems),GRAPH_TRAIN_IMAGES)]
    X=[];Y=[];records=[]
    for stem in tqdm(stems,desc='Train-only pair labels from permitted annotations'):
        rr,ed,ft=enumerate_pairs(stem,TRAIN)
        if not ft:continue
        y=pair_truth(stem,rr,ed)
        X.extend(ft);Y.extend(map(int,y))
        records.append({'stem':stem,'pairs':len(y),'positives':int(y.sum())})
    frame=pd.DataFrame(records);frame.to_csv(PAIRS_CSV,index=False)
    print('Pair data:',len(Y),'positive:',sum(Y),'image count:',len(frame))
    TRAIN_MODEL_AVAILABLE=(len(Y)>=60 and sum(Y)>=10 and len(Y)-sum(Y)>=20)
    if TRAIN_MODEL_AVAILABLE:
        X=np.asarray(X,np.float32);Y=np.asarray(Y,np.uint8)
        model=HistGradientBoostingClassifier(max_iter=130,learning_rate=.05,max_leaf_nodes=9,
                   max_depth=4,min_samples_leaf=15,l2_regularization=5,random_state=SEED)
        positives=max(1,int(Y.sum()));negatives=int((Y==0).sum())
        weights=np.where(Y==1,min(12.,negatives/positives),1.)
        model.fit(X,Y,sample_weight=weights)
        joblib.dump(model,QFILE)
        QMETA.write_text(json.dumps({'model_sha':sha(QFILE),'features':FEAT_NAMES,'splitsha':SPLIT_SHA,
           'checkpoint_sha':CKPT_SHA,'train_stems':stems,'training_pairs':len(Y),'positives':positives},indent=2))
    else:
        print('Not enough positive pair labels. Safely disabling graph; keep baseline.')
        model=None
else:
    source=QFILE if QFILE.is_file() else PREV/'pair_graph_model.pkl' if PREV else None
    model=joblib.load(source) if source and source.is_file() else None
print('Graph model:',type(model).__name__ if model is not None else 'UNAVAILABLE')

## 7 · Greedy connected-component graph merging, *without* duplicating or overlapping masks

Merging unions two entire existing instances only when an edge exceeds calibrated probability. The merged RLE need not be geometrically connected: this tests instance identity without hallucinating an image bridge. That is a conservative starting point for topology repair.

In [ ]:
MERGE_MEM={}
def graph_merge(stem,lookup,threshold=.75,max_gap=45,max_merges=6):
    rs,edges,feat=enumerate_pairs(stem,lookup)
    if model is None or len(rs)<2 or not feat:return rs
    key=(stem,'test' if lookup is TEST else 'train',threshold,max_gap,max_merges)
    if key in MERGE_MEM:return MERGE_MEM[key]
    probs=model.predict_proba(np.asarray(feat,np.float32))[:,1]
    candidates=[]
    for (i,j,_),fv,pr in zip(edges,feat,probs):
        if fv[0]<=max_gap and pr>=threshold:candidates.append((float(pr),i,j))
    candidates.sort(reverse=True)
    # Union-find; cap merge count and group size to avoid overmerging.
    parent=list(range(len(rs)));size=[1]*len(rs)
    def root(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]];i=parent[i]
        return i
    merges=0
    for pr,i,j in candidates:
        a,b=root(i),root(j)
        if a==b or size[a]+size[b]>3:continue
        parent[b]=a;size[a]+=size[b];merges+=1
        if merges>=max_merges:break
    groups=defaultdict(list)
    for i in range(len(rs)):groups[root(i)].append(i)
    output=[]
    for ids in groups.values():
        if len(ids)==1:output.append(rs[ids[0]])
        else:
            # Source masks are disjoint; exact union is cheap in binary/COCO.
            output.append(coco.merge([rs[i] for i in ids],intersect=False))
    assert len(output)<=len(rs) and disjoint(output)
    MERGE_MEM[key]=output
    return output

## 8 · Frozen selection on reserved tune images, historical-audit diagnostics

Selection uses the 47 tune observations never used to reconstruct V7. The classifier was trained only on V4's original 513 TRAIN images. The previous 97-image audit is **not** an independent holdout; report it but do not optimize on it.

In [ ]:
SELECT_FILE=OUT/'selection.json';RESULT_FILE=OUT/'selection_results.csv'
if MODE=='SUBMIT':
    p=SELECT_FILE if SELECT_FILE.is_file() else PREV/'selection.json' if PREV else None
    if p is None or not p.is_file():raise FileNotFoundError('Missing selected graph configuration. Run FIT first.')
    DECISION=json.loads(p.read_text())
else:
    old=score(SELECT,lambda s:baseline(s,TRAIN))
    results=[{'threshold':None,'max_gap':None,'max_merges':0,'method':'baseline',**old}]
    if model is not None:
        for threshold,gap,cap in tqdm(list(product((.60,.75,.87), (25,50,85),(3,6))),desc='Reserved-tune graph calibration'):
            met=score(SELECT,lambda s:graph_merge(s,TRAIN,threshold,gap,cap))
            results.append({'threshold':threshold,'max_gap':gap,'max_merges':cap,'method':'graph',**met})
    frame=pd.DataFrame(results).sort_values(['pq','rq'],ascending=False)
    frame.to_csv(RESULT_FILE,index=False)
    display(frame.head(12))
    candidate=frame.iloc[0];gain=float(candidate['pq']-old['pq'])
    promote=(candidate['method']=='graph' and gain>=MIN_ABSOLUTE_GAIN)
    chosen={'mode':'graph','threshold':float(candidate['threshold']),'max_gap':int(candidate['max_gap']),
            'max_merges':int(candidate['max_merges'])} if promote else {'mode':'baseline'}
    DECISION={'chosen':chosen,'validation_pq':float(candidate['pq']),
       'baseline_pq':float(old['pq']),'gain':gain,'promotion_margin':MIN_ABSOLUTE_GAIN,
       'identity':V7_PICK['identity'],'splitsha':SPLIT_SHA,'modelsha':CKPT_SHA,
       'pair_model_sha':sha(QFILE) if QFILE.exists() else None,
       'base_fit_stems':BASE_FIT,'graph_selection_stems':SELECT,
       'audit_is_not_independent':True}
    SELECT_FILE.write_text(json.dumps(DECISION,indent=2))
    print('Frozen decision:',chosen,'gain:',gain)
    historical=sorted(SPLIT['audit'])
    # Previously seen audit: no parameter selection / threshold tuning.
    oldaudit=score(historical,lambda s:baseline(s,TRAIN))
    selectedaudit=score(historical,lambda s:graph_merge(s,TRAIN,**{k:v for k,v in chosen.items() if k!='mode'}) if chosen['mode']=='graph' else baseline(s,TRAIN))
    (OUT/'historical_audit.json').write_text(json.dumps({'status':'PREVIOUSLY_SEEN_NOT_INDEPENDENT',
           'baseline':oldaudit,'selected':selectedaudit,'delta_pq':selectedaudit['pq']-oldaudit['pq']},indent=2))
assert DECISION['splitsha']==SPLIT_SHA and DECISION['modelsha']==CKPT_SHA
print('V11 selection:',DECISION['chosen'],'graph heldout gain',DECISION['gain'])

## 9 · Optional CSV export: *only* after a demonstrated held-out gain

Export requires the graph to beat the baseline by `MIN_ABSOLUTE_GAIN`; when not promoted, default behavior is to **stop instead of submitting an unchanged 0.33 baseline**. If you explicitly want that fallback, change `ALLOW_BASELINE_EXPORT=True` in the configuration. Use Kaggle to score only genuine live-image inference.

In [ ]:
def chosen_masks(stem,lookup):
    cfg=DECISION['chosen']
    if cfg['mode']=='baseline':return baseline(stem,lookup)
    return graph_merge(stem,lookup,cfg['threshold'],cfg['max_gap'],cfg['max_merges'])
def export_test():
    if DECISION['chosen']['mode']!='graph' and not ALLOW_BASELINE_EXPORT:
        print('NO SUBMISSION: V11 did not beat baseline on reserved validation. Preserve your 0.34 entry.')
        return None
    if DECISION['chosen']['mode']=='graph' and model is None:raise RuntimeError('Graph model missing')
    rows=[];counts=[]
    for stem in tqdm(sorted(TEST),desc='CPU genuine test inference'):
        pred=chosen_masks(stem,TEST)
        assert disjoint(pred),(stem,'overlapping masks')
        for idx,rr in enumerate(pred,1):
            if coco.area(rr)<=0:raise ValueError(stem)
            rows.append({'filament_id':f'{stem}_{idx}','segmentation_rle':asc(rr)})
        counts.append({'stem':stem,'count':len(pred)})
    path=Path('/kaggle/working/submission.csv' if Path('/kaggle/working').exists() else OUT/'submission.csv')
    df=pd.DataFrame(rows,columns=['filament_id','segmentation_rle'])
    assert not df.empty and df.filament_id.is_unique
    df.to_csv(path,index=False)
    chk=pd.read_csv(path,keep_default_na=False)
    assert len(chk)==len(df) and list(chk.columns)==['filament_id','segmentation_rle']
    result={'path':str(path),'sha256':sha(path),'rows':len(df),'images_with_masks':sum(x['count']>0 for x in counts),
            'strategy':DECISION['chosen'],'baseline_identity':V7_PICK['identity'],
            'checkpoint_sha':CKPT_SHA,'no_embedded_test_predictions':True}
    (OUT/'submission_report.json').write_text(json.dumps(result,indent=2))
    pd.DataFrame(counts).to_csv(OUT/'test_instance_counts.csv',index=False)
    print('Submission:',json.dumps(result,indent=2))
    return path
if MODE in ('SUBMIT','ALL'):export_test()
else:print('FIT completed. Review selection_results.csv; export only if gain is meaningful.')

## 10 · Interpretation and next experiments

- **Graph wins** by at least +0.015 absolute PQ on reserved tune → optionally run `SUBMIT`, keeping the original 0.34 CSV safe.
- **Graph ties or loses** → do **not** send another baseline-equivalent Kaggle submission. Save `train_pair_diagnostics.csv` and `selection_results.csv` to diagnose whether fragmentation was actually the right target.
- **Few positive graph pairs** → the current detector may be producing misses, not two pieces of the same filament. A stronger *new model* is then necessary, ideally native-resolution soft-consensus U-Net / DETR-to-crop refiner on a GPU when one becomes available. CPU-only graph repair cannot replace unseen filaments.
- **V7 exact recipe missing** → attach the saved `solar_cpu_v7/selection.json` or the historic 0.34 output to establish a reproducible control. A reconstructed model can score differently, as V10 demonstrated.

Competition requires the model, training and inference code and reproducible documentation for prize eligibility: https://www.kaggle.com/competitions/filament-segmentation-2026/rules .